# 04 – Analysis & Visualisation
Produce all publication-quality plots and summary statistics.

**Inputs** : topic model outputs, interaction data, engagement spreadsheet  
**Outputs**: figures saved to `outputs/plots/`

In [ ]:
import logging, sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loaders import load_config, load_celebrity_engagement, load_document_topics
from src.visualization.plots import (
    plot_topic_freq_by_party,
    plot_monthly_trends,
    plot_partisanship_histograms,
    plot_language_distribution,
    plot_topic_bucket_distribution,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
cfg = load_config('configs/config.yaml')
palette      = cfg['viz']['palette']
pal_interact = cfg['viz']['interaction_palette']
plots_dir    = cfg['outputs']['plots_dir']
os.makedirs(plots_dir, exist_ok=True)
print('Setup complete.')

## 1. Politician topic frequency by party and category

In [ ]:
df_bjp = load_document_topics('BJP', 'politicians', cfg)
df_inc = load_document_topics('INC', 'politicians', cfg)
df_pol = pd.concat([df_bjp, df_inc], ignore_index=True)

# Normalise label
df_pol.loc[df_pol['topic_label'] == 'attack', 'topic_label'] = 'non-greetings'

df_group = (
    df_pol
    .groupby(['party', 'category', 'topic_label'], as_index=False)
    .agg({'id': 'count'})
    .rename(columns={'id': 'freq'})
)
df_group.head()

In [ ]:
for cat in ['sports', 'entertainment']:
    plot_topic_freq_by_party(
        df_group, category=cat, palette=palette,
        title=f'Politicians – {cat}',
        out_path=f'{plots_dir}politicians_{cat}.png',
        dpi=cfg['viz']['dpi'],
    )

## 2. Monthly politician tweet frequency

In [ ]:
df_pol['month'] = pd.to_datetime(df_pol['date']).dt.to_period('M')

df_monthly = (
    df_pol
    .groupby(['month', 'category', 'topic_label', 'party'], as_index=False)
    .agg({'id': 'count'})
    .rename(columns={'id': 'freq'})
)

# Fill missing month×party×category×label combinations with 0
full_index = pd.MultiIndex.from_product(
    [df_monthly['month'].unique(), df_monthly['party'].unique(),
     df_monthly['category'].unique(), df_monthly['topic_label'].unique()],
    names=['month', 'party', 'category', 'topic_label'],
)
df_monthly = (
    df_monthly
    .set_index(['month', 'party', 'category', 'topic_label'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
df_monthly['month'] = df_monthly['month'].astype(str)
df_monthly.head()

In [ ]:
for cat in ['sports', 'entertainment']:
    plot_monthly_trends(
        df_monthly, category=cat, palette=palette,
        title=f'Politicians monthly – {cat}',
        out_path=f'{plots_dir}politicians_{cat}_monthly.png',
        dpi=cfg['viz']['dpi'],
    )

## 3. Celebrity topic frequency

In [ ]:
df_cbjp = load_document_topics('BJP', 'celebs', cfg)
df_cinc = load_document_topics('INC', 'celebs', cfg)
df_cel = pd.concat([df_cbjp, df_cinc], ignore_index=True)

attack_labels = {'attack', 'attack_BJP', 'attack_INC', 'movie_releases', 'support_BJP', 'support_INC'}
df_cel.loc[df_cel['topic_label'].isin(attack_labels), 'topic_label'] = 'non-greetings'

df_cel_group = (
    df_cel
    .groupby(['party', 'category', 'topic_label'], as_index=False)
    .agg({'id': 'count'})
    .rename(columns={'id': 'freq'})
)

for cat in ['sports', 'entertainment']:
    plot_topic_freq_by_party(
        df_cel_group, category=cat, palette=palette,
        title=f'Celebrities – {cat}',
        out_path=f'{plots_dir}celebs_{cat}.png',
        dpi=cfg['viz']['dpi'],
    )

## 4. Partisanship histogram

In [ ]:
engagement_df = load_celebrity_engagement(cfg)

# Filter out celebs with zero engagement with both parties
no_eng = engagement_df[
    (engagement_df['eng_with_bjp_overall'] == 0) &
    (engagement_df['eng_with_inc_overall'] == 0)
]['screen_name'].tolist()

ttype_cols = ['rt', 'quote', 'mention', 'reply']
base_cols  = ['screen_name', 'name', 'gender', 'vocation', 'followers_count']
eng_cols   = [f'eng_with_{p}_{t}' for p in ['bjp', 'inc'] for t in ttype_cols]
plotdf = engagement_df[~engagement_df['screen_name'].isin(no_eng)][base_cols + eng_cols].copy()

for ttype in ttype_cols:
    plotdf[f'partisanship_{ttype}'] = (
        plotdf[f'eng_with_bjp_{ttype}'] - plotdf[f'eng_with_inc_{ttype}']
    )

print(f'Plotting {len(plotdf)} celebrities')

In [ ]:
plot_partisanship_histograms(
    plotdf,
    interaction_types=['rt', 'quote', 'mention', 'reply'],
    vocations=['entertainment', 'sports'],
    out_path=f'{plots_dir}partisanship_by_gender.png',
    ylims={'entertainment': 90, 'sports': 35},
    dpi=cfg['viz']['dpi'],
)

## 5. Language distribution

In [ ]:
def get_lang_label(code):
    return {'hi': 'hindi', 'en': 'english'}.get(code, 'other')

lang_data = {}
titles = []
for group in ['celebs', 'politicians']:
    for party in ['INC', 'BJP']:
        key = f'{group}/{party}'
        topic_dir = os.path.join(cfg['outputs']['topic_models_dir'], group, party)
        path = os.path.join(topic_dir, 'rt_document_topics.json')
        if not os.path.exists(path):
            continue
        df_t = pd.read_json(path)
        df_t['language']   = df_t['lang'].apply(get_lang_label)
        df_t['tweet_type'] = df_t['type'].apply(lambda x: 'retweet' if x == 'rt' else x)
        df_agg = (
            df_t.groupby(['language', 'tweet_type'], as_index=False)
            .agg({'target': 'count'})
            .rename(columns={'target': 'frequency'})
        )
        lang_data[key] = df_agg
        titles.append(f'{group.capitalize()} → {party}')

if lang_data:
    plot_language_distribution(
        lang_data, titles=titles, pal=pal_interact,
        out_path=f'{plots_dir}tweet_distribution_language.png',
        dpi=cfg['viz']['dpi'],
    )
else:
    print('No topic-model outputs found yet – run notebook 02 first.')